#Ciência de Dados: Análise de Dados Aplicada
##Trabalho final


###Aluno: Alan Nakamura Kageyama
###Ra: 2482053

##Bibliotecas

In [ ]:
import urllib.request
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

import duckdb

##Fonte Principal: NYC Citywide Annualized Calendar Sales Update

Os dados dos anos de 2018-2021 são muito complicados por causa da pandemia. Por conta disso, decidi pegar os anos de 2022-2025.

###Carregando os dados

In [ ]:
print("Iniciando o download do dataset Principal. Isso pode levar alguns segundos!")

filename = "nyc_sales_raw.csv"
url = "https://data.cityofnewyork.us/api/views/w2pb-icbu/rows.csv?accessType=DOWNLOAD"

if not os.path.exists(filename):
    try:
        urllib.request.urlretrieve(url, filename)
        tamanho_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"{filename} baixado com sucesso! Tamanho: {tamanho_mb:.2f} MB")
    except Exception as e:
        print(f"Erro ao baixar {filename}: {e}")
else:
    tamanho_mb = os.path.getsize(filename) / (1024 * 1024)
    print(f"O arquivo {filename} já existe no diretório. Tamanho: {tamanho_mb:.2f} MB.")

print("Download foi finalizado!")

Iniciando o download do dataset Principal. Isso pode levar alguns segundos!
O arquivo nyc_sales_raw.csv já existe no diretório. Tamanho: 148.20 MB.
Download foi finalizado!


###Analisando os dados

In [ ]:
df_nyc_sales = pd.read_csv('nyc_sales_raw.csv')
display(df_nyc_sales.head())

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AS OF FINAL ROLL,BLOCK,LOT,EASE-MENT,BUILDING CLASS AS OF FINAL ROLL,ADDRESS,APARTMENT NUMBER,...,SALE PRICE,SALE DATE,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract 2020,Neighborhood Tabulation Area (NTA) (2020)
0,1,CHELSEA,21 OFFICE BUILDINGS,4,697,5,NaN,O2,555 WEST 25TH STREET,NaN,...,43300000,03/28/2019,40.749704,-74.004930,104.0,3.0,1012379.0,1.006970e+09,9901.0,MN0401
1,1,CHELSEA,21 OFFICE BUILDINGS,4,697,23,NaN,O6,511 WEST 25TH STREET,NaN,...,148254147,05/23/2019,40.749364,-74.004132,104.0,3.0,1012382.0,1.006970e+09,9901.0,MN0401
2,1,CHELSEA,21 OFFICE BUILDINGS,4,700,55,NaN,O2,538 WEST 29TH STREET,NaN,...,11000000,03/13/2019,40.752067,-74.002931,104.0,3.0,1012435.0,1.007000e+09,9902.0,MN0401
3,1,CHELSEA,21 OFFICE BUILDINGS,4,712,1,NaN,O6,450 WEST 15TH,NaN,...,591800000,05/22/2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,CHELSEA,21 OFFICE BUILDINGS,4,746,64,NaN,O8,340 WEST 23RD STREET,NaN,...,0,04/01/2019,40.745809,-73.999729,104.0,3.0,1013367.0,1.007460e+09,93.0,MN0401


In [ ]:
display(df_nyc_sales.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 845845 entries, 0 to 845844
Data columns (total 29 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   BOROUGH                                    845845 non-null  int64  
 1   NEIGHBORHOOD                               845845 non-null  object 
 2   BUILDING CLASS CATEGORY                    845845 non-null  object 
 3   TAX CLASS AS OF FINAL ROLL                 807448 non-null  object 
 4   BLOCK                                      845845 non-null  int64  
 5   LOT                                        845845 non-null  int64  
 6   EASE-MENT                                  0 non-null       float64
 7   BUILDING CLASS AS OF FINAL ROLL            807448 non-null  object 
 8   ADDRESS                                    845845 non-null  object 
 9   APARTMENT NUMBER                           200491 non-null  object 
 10  ZIP CODE

None

In [ ]:
display(df_nyc_sales.isnull().sum())

,0
BOROUGH,0
NEIGHBORHOOD,0
BUILDING CLASS CATEGORY,0
TAX CLASS AS OF FINAL ROLL,38397
BLOCK,0
LOT,0
EASE-MENT,845845
BUILDING CLASS AS OF FINAL ROLL,38397
ADDRESS,0
APARTMENT NUMBER,645354


###Limpeza de dados antes da integração

####Colunas para dropar
EASE-MENT: Coluna está completamente vazia.

APARTMENT NUMBER: Como estou analisando o impacto do bairro e indicadores macroeconômicos, o número exato do apartamento é um excesso de ruído.

BLOCK e LOT: Usarei o BOROUGH para integrar com os crimes. Deixar esses no modelo imagino que pode causar overfitting.

As outras são para questões administrativas, então podem ser retiradas.

In [ ]:
columns_to_drop = ['EASE-MENT', 'APARTMENT NUMBER', 'BLOCK', 'LOT', 'Community Board',
                   'Council District', 'BIN', 'BBL', 'Census Tract 2020', 'Neighborhood Tabulation Area (NTA) (2020)',
                   'TAX CLASS AS OF FINAL ROLL', 'BUILDING CLASS AS OF FINAL ROLL']
df_nyc_sales = df_nyc_sales.drop(columns=columns_to_drop)

####Linhas para dropar, outliers e valores nulos
SALE_PRICE: Nula ou menor que 10.000, valores de 0 a 10.000 geralmente são trocas que não envolvem dinheiro ou heranças, e não vendas reais a preço de mercado.

SALE_DATE: Só é necessário o periodo de 2022-2025

ZIP_CODE: Nulo ou igual a 0, não é possivel integrar com a tabela de crimes nem descobrir se a escola vizinha é boa.

Latitude e Longitude são dados interessantes de ter, por isso vou eliminar as linhas com valores nulos.

Vendas duplicadas: Foram identificados algumas vendas duplicadas.

GROSS SQUARE FEET: Algumas linhas possuem valores muito baixos.

In [ ]:
linhas_antes = df_nyc_sales.shape[0]

df_nyc_sales.dropna(subset=['SALE PRICE'], inplace=True)
df_nyc_sales = df_nyc_sales[df_nyc_sales['SALE PRICE'] > 10000]

df_nyc_sales.dropna(subset=['ZIP CODE'], inplace=True)
df_nyc_sales = df_nyc_sales[df_nyc_sales['ZIP CODE'] > 0]

df_nyc_sales.dropna(subset=['Latitude', 'Longitude'], inplace=True)

df_nyc_sales.drop_duplicates(inplace=True)

df_nyc_sales['SALE DATE'] = pd.to_datetime(df_nyc_sales['SALE DATE'])
df_nyc_sales = df_nyc_sales[df_nyc_sales['SALE DATE'].dt.year.between(2022, 2025)]

In [ ]:
area_minima = 100

df_nyc_sales['GROSS SQUARE FEET'] = pd.to_numeric(
    df_nyc_sales['GROSS SQUARE FEET'].astype(str).str.replace(',', ''),
    errors='coerce'
)
df_nyc_sales.dropna(subset=['GROSS SQUARE FEET'], inplace=True)


df_nyc_sales = df_nyc_sales[
    (df_nyc_sales['GROSS SQUARE FEET'] >= area_minima)
].copy()

linhas_depois = df_nyc_sales.shape[0]
print(f"linhas removidas: {linhas_antes - linhas_depois} linhas descartadas.")

linhas removidas: 760877 linhas descartadas.


In [ ]:
print(f"New shape of df_nyc_sales: {df_nyc_sales.shape}")

New shape of df_nyc_sales: (84968, 17)


In [ ]:
display(df_nyc_sales.isnull().sum())

,0
BOROUGH,0
NEIGHBORHOOD,0
BUILDING CLASS CATEGORY,0
ADDRESS,0
ZIP CODE,0
RESIDENTIAL UNITS,0
COMMERCIAL UNITS,0
TOTAL UNITS,0
LAND SQUARE FEET,0
GROSS SQUARE FEET,0


####Padronizando os dados que seram usados para a integração
Aqui o formato após a conversão é string (YYYY-MM)

Garantindo que o borough ta no formato certo

In [ ]:
df_nyc_sales['SALE DATE'] = df_nyc_sales['SALE DATE'].dt.strftime('%Y-%m')
df_nyc_sales['BOROUGH'] = df_nyc_sales['BOROUGH'].astype(int)

In [ ]:
if not os.path.exists('dados'):
    os.makedirs('dados')

df_nyc_sales.to_parquet('dados/nyc_vendas_limpo_sem_integracao.parquet', index=False)
print("Dataset de vendas limpo sem integração salvo com sucesso na pasta 'dados'!")

Dataset de vendas limpo sem integração salvo com sucesso na pasta 'dados'!


##Enriquecimento com Fontes Externa

###NYPD Complaint Data Historic

####Carregando os dados

In [ ]:
print("Iniciando o download do dataset NYPD Complaint Data Historic. Arquivo muito grande! Isso pode levar alguns minutos dependendo da sua internet...")

filename = "nypd_crimes_raw.csv"
url = "https://data.cityofnewyork.us/api/views/qgea-i56i/rows.csv?accessType=DOWNLOAD"

if not os.path.exists(filename):
    try:
        urllib.request.urlretrieve(url, filename)
        tamanho_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"{filename} baixado com sucesso! Tamanho: {tamanho_mb:.2f} MB")
    except Exception as e:
        print(f"Erro ao baixar {filename}: {e}")
else:
    tamanho_mb = os.path.getsize(filename) / (1024 * 1024)
    print(f"O arquivo {filename} já existe no diretório. Tamanho: {tamanho_mb:.2f} MB.")

print("Download foi finalizado!")

Iniciando o download do dataset NYPD Complaint Data Historic. Arquivo muito grande! Isso pode levar alguns minutos dependendo da sua internet...
O arquivo nypd_crimes_raw.csv já existe no diretório. Tamanho: 3270.30 MB.
Download foi finalizado!


####Analisando os dados

In [ ]:
con = duckdb.connect()

#####Reduzindo o Dataset para apenas o periodo de 2022-2025

In [ ]:
print("Lendo o arquivo e filtrando apenas os anos de 2022 a 2025...")

query = """
    SELECT
        RPT_DT,
        BORO_NM,
        LAW_CAT_CD
    FROM read_csv_auto('nypd_crimes_raw.csv', all_varchar=true)
    WHERE
        CAST(str_split(RPT_DT, '/')[3] AS INTEGER) BETWEEN 2022 AND 2025
"""

df_crimes_filtrados = con.execute(query).df()

print(f"Sucesso! O dataset filtrado tem {df_crimes_filtrados.shape[0]} linhas.")

Lendo o arquivo e filtrando apenas os anos de 2022 a 2025...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sucesso! O dataset filtrado tem 2243782 linhas.


In [ ]:
display(df_crimes_filtrados.head())

,RPT_DT,BORO_NM,LAW_CAT_CD
0,03/15/2022,BRONX,FELONY
1,03/18/2022,BRONX,FELONY
2,06/23/2022,BROOKLYN,FELONY
3,08/20/2022,BRONX,VIOLATION
4,03/08/2022,BRONX,VIOLATION


In [ ]:
display(df_crimes_filtrados.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2243782 entries, 0 to 2243781
Data columns (total 3 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   RPT_DT      object
 1   BORO_NM     object
 2   LAW_CAT_CD  object
dtypes: object(3)
memory usage: 51.4+ MB


None

In [ ]:
display(df_crimes_filtrados.isnull().sum())

,0
RPT_DT,0
BORO_NM,0
LAW_CAT_CD,0


####Limpando e padronizando os dados

In [ ]:
#Data que ocorreu e o nome do distrito
df_crimes = df_crimes_filtrados.dropna(subset=['RPT_DT', 'BORO_NM']).copy()

dict_boroughs = {
    'MANHATTAN': 1,
    'BRONX': 2,
    'BROOKLYN': 3,
    'QUEENS': 4,
    'STATEN ISLAND': 5
}
df_crimes['borough_code'] = df_crimes['BORO_NM'].map(dict_boroughs)

df_crimes.dropna(subset=['borough_code'], inplace=True)

# Criando a chave Ano-Mês no formato YYYY-MM
df_crimes['RPT_DT'] = pd.to_datetime(df_crimes['RPT_DT'], errors='coerce')
df_crimes.dropna(subset=['RPT_DT'], inplace=True)
df_crimes['ano_mes'] = df_crimes['RPT_DT'].dt.strftime('%Y-%m')

#Contando quantos crimes graves e leves ocorreram por Bairro e por Mês
df_crimes_preparado = df_crimes.groupby(['borough_code', 'ano_mes']).apply(
    lambda x: pd.Series({
        'qtd_crimes_graves': (x['LAW_CAT_CD'] == 'FELONY').sum(),
        'qtd_crimes_leves': (x['LAW_CAT_CD'] == 'MISDEMEANOR').sum()
    })
).reset_index()

df_crimes_preparado['borough_code'] = df_crimes_preparado['borough_code'].astype(int)

print(f"Total de linhas agregadas: {df_crimes_preparado.shape[0]}")
display(df_crimes_preparado.head())

Total de linhas agregadas: 240


/tmp/ipykernel_1871/780189284.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_crimes_preparado = df_crimes.groupby(['borough_code', 'ano_mes']).apply(


,borough_code,ano_mes,qtd_crimes_graves,qtd_crimes_leves
0,1,2022-01,3262,4738
1,1,2022-02,3284,5343
2,1,2022-03,3647,6308
3,1,2022-04,3664,5956
4,1,2022-05,3971,6222


In [ ]:
if not os.path.exists('dados'):
    os.makedirs('dados')

df_crimes_preparado.to_parquet('dados/crimes_agregados_mensal.parquet', index=False)
print("Arquivo salvo com sucesso na pasta 'dados'!")

Arquivo salvo com sucesso na pasta 'dados'!


###FRED - 30-Year Fixed Rate Mortgage Average

####Carregando os dados

In [ ]:
print("Iniciando o download do dataset 30-Year Fixed Rate Mortgage Average.")

filename = "macro_economics_usa.csv"
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

if not os.path.exists(filename):
    try:
        urllib.request.urlretrieve(url, filename)
        tamanho_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"{filename} baixado com sucesso! Tamanho: {tamanho_mb:.2f} MB")
    except Exception as e:
        print(f"Erro ao baixar {filename}: {e}")
else:
    tamanho_mb = os.path.getsize(filename) / (1024 * 1024)
    print(f"O arquivo {filename} já existe no diretório. Tamanho: {tamanho_mb:.2f} MB.")

print("Download foi finalizado!")

Iniciando o download do dataset 30-Year Fixed Rate Mortgage Average.
O arquivo macro_economics_usa.csv já existe no diretório. Tamanho: 0.04 MB.
Download foi finalizado!


####Analisando os dados

In [ ]:
df_juros = pd.read_csv('macro_economics_usa.csv')

display(df_juros.head())

,observation_date,MORTGAGE30US
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [ ]:
df_juros['observation_date'] = pd.to_datetime(df_juros['observation_date'], errors='coerce')

# filtrando o dataset para os anos 2022-2025
df_juros = df_juros[df_juros['observation_date'].dt.year.between(2022, 2025)]

df_juros['ano_mes'] = df_juros['observation_date'].dt.strftime('%Y-%m')

display(df_juros.head())

,observation_date,MORTGAGE30US,ano_mes
2649,2022-01-06,3.22,2022-01
2650,2022-01-13,3.45,2022-01
2651,2022-01-20,3.56,2022-01
2652,2022-01-27,3.55,2022-01
2653,2022-02-03,3.55,2022-02


Padronização dos dados para integração

In [ ]:
df_juros_preparado = df_juros.groupby('ano_mes')['MORTGAGE30US'].mean().reset_index()

df_juros_preparado.rename(columns={'MORTGAGE30US': 'media_juros_hipoteca'}, inplace=True)

display(df_juros_preparado.head())

,ano_mes,media_juros_hipoteca
0,2022-01,3.4450
1,2022-02,3.7625
2,2022-03,4.1720
3,2022-04,4.9825
4,2022-05,5.2300


###Integrando ao Dataset principal

A integração vai ser pelo codigo de distritos(borough) e mês/ano. Como no dataset principal, o distrito era identificado por id, foi necessário transformar o borough desse dataset em código.

In [ ]:
# Bairro e o mês de venda com o bairro e o mês do crime
df_integrado_1 = pd.merge(
    df_nyc_sales,
    df_crimes_preparado,
    left_on=['BOROUGH', 'SALE DATE'],
    right_on=['borough_code', 'ano_mes'],
    how='left'
)

# 'borough_code' ficou duplicado
df_integrado_1.drop(columns=['borough_code'], errors='ignore', inplace=True)

df_dataset_final = pd.merge(
    df_integrado_1,
    df_juros_preparado,
    on='ano_mes',
    how='left'
)

print(f"Integração concluída! O formato final do dataset é: {df_dataset_final.shape}")

Integração concluída! O formato final do dataset é: (84968, 21)


In [ ]:
display(df_dataset_final.head())

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,ADDRESS,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,...,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE,Latitude,Longitude,ano_mes,qtd_crimes_graves,qtd_crimes_leves,media_juros_hipoteca
0,2,BATHGATE,01 ONE FAMILY DWELLINGS,4447 PARK AVENUE,10457.0,1.0,0.0,1.0,"1,668",1497.0,...,1,A1,615000,2024-12,40.853515,-73.896028,2024-12,3269,5063,6.7150
1,2,BATHGATE,01 ONE FAMILY DWELLINGS,2073 BATHGATE AVENUE,10457.0,1.0,0.0,1.0,"1,933",1344.0,...,1,A1,425000,2024-06,40.850429,-73.895169,2024-06,3762,5154,6.9175
2,2,BATHGATE,01 ONE FAMILY DWELLINGS,503 EAST 182 STREET,10457.0,1.0,0.0,1.0,"1,960",1705.0,...,1,A9,515000,2024-04,40.853532,-73.893581,2024-04,3398,4884,6.9925
3,2,BATHGATE,01 ONE FAMILY DWELLINGS,2327 BASSFORD AVE,10458.0,1.0,0.0,1.0,913,1248.0,...,1,A1,413000,2024-08,40.855847,-73.891755,2024-08,3948,5356,6.5000
4,2,BATHGATE,02 TWO FAMILY DWELLINGS,454 EAST 172ND ST,10457.0,2.0,0.0,2.0,"1,658",1428.0,...,1,B9,500000,2024-02,40.840464,-73.902576,2024-02,3219,4805,6.7760


In [ ]:
display(df_dataset_final.isnull().sum())

,0
BOROUGH,0
NEIGHBORHOOD,0
BUILDING CLASS CATEGORY,0
ADDRESS,0
ZIP CODE,0
RESIDENTIAL UNITS,0
COMMERCIAL UNITS,0
TOTAL UNITS,0
LAND SQUARE FEET,0
GROSS SQUARE FEET,0


In [ ]:
df_dataset_final.to_parquet('dados/nyc_vendas_integrado_bruto.parquet', index=False)

##Limpeza de Dados

###Missing values

####Aplicando KNN-Imputer

In [ ]:
from tqdm.notebook import tqdm #adição da barrinha de progresso

for col in ['GROSS SQUARE FEET', 'LAND SQUARE FEET']:
    df_dataset_final[col] = pd.to_numeric(df_dataset_final[col].astype(str).str.replace(',', ''), errors='coerce')

pedacos_imputados = []

#aqui fiz por bairro apenas para poder acompanhar o progresso
for bairro, df_bairro in tqdm(df_dataset_final.groupby('BOROUGH'), desc="Processando Bairros"):

    colunas_knn = ['SALE PRICE', 'Latitude', 'Longitude', 'YEAR BUILT', 'GROSS SQUARE FEET', 'LAND SQUARE FEET']
    df_num = df_bairro[colunas_knn].copy()

    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_num), columns=df_num.columns, index=df_num.index)

    imputer = KNNImputer(n_neighbors=5, weights='distance')
    df_imputed_scaled = pd.DataFrame(imputer.fit_transform(df_scaled), columns=df_num.columns, index=df_num.index)

    df_imputed = pd.DataFrame(scaler.inverse_transform(df_imputed_scaled), columns=df_num.columns, index=df_num.index)

    df_bairro_pronto = df_bairro.copy()
    for col in colunas_knn:
        df_bairro_pronto[col] = df_imputed[col]

    pedacos_imputados.append(df_bairro_pronto)

df_dataset_final = pd.concat(pedacos_imputados).sort_index()

print("Imputação com KNN concluída")

Processando Bairros:   0%|          | 0/5 [00:00<?, ?it/s]

Imputação com KNN concluída


In [ ]:
display(df_dataset_final.isnull().sum())
df_dataset_final.to_parquet('dados/nyc_dados_finais_limpos.parquet', index=False)

,0
BOROUGH,0
NEIGHBORHOOD,0
BUILDING CLASS CATEGORY,0
ADDRESS,0
ZIP CODE,0
RESIDENTIAL UNITS,0
COMMERCIAL UNITS,0
TOTAL UNITS,0
LAND SQUARE FEET,0
GROSS SQUARE FEET,0


In [ ]:
display(df_dataset_final.head())

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,ADDRESS,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,...,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE,Latitude,Longitude,ano_mes,qtd_crimes_graves,qtd_crimes_leves,media_juros_hipoteca
0,2,BATHGATE,01 ONE FAMILY DWELLINGS,4447 PARK AVENUE,10457.0,1.0,0.0,1.0,1668.0,1497.0,...,1,A1,615000.0,2024-12,40.853515,-73.896028,2024-12,3269,5063,6.7150
1,2,BATHGATE,01 ONE FAMILY DWELLINGS,2073 BATHGATE AVENUE,10457.0,1.0,0.0,1.0,1933.0,1344.0,...,1,A1,425000.0,2024-06,40.850429,-73.895169,2024-06,3762,5154,6.9175
2,2,BATHGATE,01 ONE FAMILY DWELLINGS,503 EAST 182 STREET,10457.0,1.0,0.0,1.0,1960.0,1705.0,...,1,A9,515000.0,2024-04,40.853532,-73.893581,2024-04,3398,4884,6.9925
3,2,BATHGATE,01 ONE FAMILY DWELLINGS,2327 BASSFORD AVE,10458.0,1.0,0.0,1.0,913.0,1248.0,...,1,A1,413000.0,2024-08,40.855847,-73.891755,2024-08,3948,5356,6.5000
4,2,BATHGATE,02 TWO FAMILY DWELLINGS,454 EAST 172ND ST,10457.0,2.0,0.0,2.0,1658.0,1428.0,...,1,B9,500000.0,2024-02,40.840464,-73.902576,2024-02,3219,4805,6.7760


###Inconsistências

In [ ]:
df_dataset_final['TOTAL UNITS'] = df_dataset_final['RESIDENTIAL UNITS'] + df_dataset_final['COMMERCIAL UNITS']

###Padronização

In [ ]:
df_dataset_final['ZIP CODE'] = df_dataset_final['ZIP CODE'].astype(int).astype(str)
df_dataset_final['NEIGHBORHOOD'] = df_dataset_final['NEIGHBORHOOD'].str.strip().str.upper()
df_dataset_final['BUILDING CLASS CATEGORY'] = df_dataset_final['BUILDING CLASS CATEGORY'].str.strip().str.upper()

if 'ano_mes' in df_dataset_final.columns:
    df_dataset_final.drop(columns=['ano_mes'], inplace=True)


##Dataset Final

In [ ]:
display(df_dataset_final.head())
display(df_dataset_final.info())

,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,ADDRESS,ZIP CODE,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE,Latitude,Longitude,qtd_crimes_graves,qtd_crimes_leves,media_juros_hipoteca
0,2,BATHGATE,01 ONE FAMILY DWELLINGS,4447 PARK AVENUE,10457,1.0,0.0,1.0,1668.0,1497.0,1899.0,1,A1,615000.0,2024-12,40.853515,-73.896028,3269,5063,6.7150
1,2,BATHGATE,01 ONE FAMILY DWELLINGS,2073 BATHGATE AVENUE,10457,1.0,0.0,1.0,1933.0,1344.0,1899.0,1,A1,425000.0,2024-06,40.850429,-73.895169,3762,5154,6.9175
2,2,BATHGATE,01 ONE FAMILY DWELLINGS,503 EAST 182 STREET,10457,1.0,0.0,1.0,1960.0,1705.0,1901.0,1,A9,515000.0,2024-04,40.853532,-73.893581,3398,4884,6.9925
3,2,BATHGATE,01 ONE FAMILY DWELLINGS,2327 BASSFORD AVE,10458,1.0,0.0,1.0,913.0,1248.0,1901.0,1,A1,413000.0,2024-08,40.855847,-73.891755,3948,5356,6.5000
4,2,BATHGATE,02 TWO FAMILY DWELLINGS,454 EAST 172ND ST,10457,2.0,0.0,2.0,1658.0,1428.0,1901.0,1,B9,500000.0,2024-02,40.840464,-73.902576,3219,4805,6.7760


<class 'pandas.core.frame.DataFrame'>
Index: 84968 entries, 0 to 84967
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   BOROUGH                         84968 non-null  int64  
 1   NEIGHBORHOOD                    84968 non-null  object 
 2   BUILDING CLASS CATEGORY         84968 non-null  object 
 3   ADDRESS                         84968 non-null  object 
 4   ZIP CODE                        84968 non-null  object 
 5   RESIDENTIAL UNITS               84968 non-null  float64
 6   COMMERCIAL UNITS                84968 non-null  float64
 7   TOTAL UNITS                     84968 non-null  float64
 8   LAND SQUARE FEET                84968 non-null  float64
 9   GROSS SQUARE FEET               84968 non-null  float64
 10  YEAR BUILT                      84968 non-null  float64
 11  TAX CLASS AT TIME OF SALE       84968 non-null  int64  
 12  BUILDING CLASS AT TIME OF SALE  84968

None

O Dataset final ficou com 84968 linhas e 20 colunas. Para fins de comparação, outro dataset das vendas sem a integração também foi criado.